# IEEE 118-Bus System — DC Optimal Power Flow (DC-OPF)

This notebook solves the DC-OPF with economic dispatch for the IEEE 118-bus system.

**Objective:** Minimize total generation cost

$$\min \sum_{g} c_g \cdot p_g$$

**Subject to:**

| Constraint | Expression |
|---|---|
| Nodal power balance | $B' \theta = C_g p - p_d$ |
| DC power flow | $f_l = b_l \sum_i A_{li} \theta_i$ |
| Branch thermal limits | $-F^{\max}_l \leq f_l \leq F^{\max}_l$ |
| Generator limits | $P^{\min}_g \leq p_g \leq P^{\max}_g$ |

All data is loaded from CSV files exported by `pre-processing.ipynb`.  
All quantities are in **per-unit** (base = 100 MVA) unless noted otherwise.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB

## 2. Load Data from CSV Files

All files were exported by `pre-processing.ipynb` (sections 7 and 8).

In [ ]:
# --- System base and slack bus ---
base_mva  = pd.read_csv("data/base_mva.csv")["base_mva"].iloc[0]   # 100 MVA
slack_bus = pd.read_csv("data/slack_bus.csv")["slack_bus_pp"].iloc[0]  # 0-indexed

print(f"System base : {base_mva} MVA")
print(f"Slack bus   : {int(slack_bus)} (pandapower) = {int(slack_bus)+1} (MATPOWER)")

In [ ]:
# --- Generator data (53 generators, slack excluded) ---
gen_df      = pd.read_csv("data/gen_data.csv")
gen_buses   = gen_df["bus_pp"].values     # 0-indexed
gen_pmin_pu = gen_df["pmin_pu"].values
gen_pmax_pu = gen_df["pmax_pu"].values
n_gen       = len(gen_buses)

print(f"Generators: {n_gen}")
gen_df.head(10)

In [ ]:
# --- Load data ---
load_df    = pd.read_csv("data/load_data.csv")
load_buses = load_df["bus_pp"].values     # 0-indexed
load_p_pu  = load_df["p_pu"].values

print(f"Loads        : {len(load_df)}")
print(f"Total demand : {load_p_pu.sum() * base_mva:.2f} MW")
load_df.head(10)

In [ ]:
# --- Branch data ---
branch_df        = pd.read_csv("data/branch_data.csv")
branch_labels    = branch_df["label"].values
branches_from    = branch_df["from_bus_pp"].values   # 0-indexed
branches_to      = branch_df["to_bus_pp"].values     # 0-indexed
branches_rate_pu = branch_df["rate_pu"].values
num_branches     = len(branch_df)

print(f"Branches: {num_branches}")
branch_df.head(10)

In [ ]:
# --- DC power flow matrices ---
A_df    = pd.read_csv("data/matrices/A_incidence.csv",              index_col=0)
B_diag  = pd.read_csv("data/matrices/B_branch_susceptance.csv",     index_col=0).values.diagonal()
A       = A_df.values
B_prime = pd.read_csv("data/matrices/B_prime_bus_susceptance.csv",  index_col=0).values
# Phi is not needed directly — the model uses theta variables instead

num_ns_buses = A.shape[1]   # 117 non-slack buses

# Bus-to-column index: maps each non-slack bus (0-indexed) to its column in A
non_slack_cols = [int(c) - 1 for c in A_df.columns]  # convert MATPOWER label → 0-indexed
bus_to_col     = {bus: idx for idx, bus in enumerate(non_slack_cols)}

print(f"A shape     : {A.shape}")
print(f"B' shape    : {B_prime.shape}")
print(f"B_diag size : {B_diag.shape[0]}")

## 3. Load Generation Cost Data

Cost coefficients come from the MATPOWER case118 (`gencost_data_118-bus.csv`).  
The cost function is polynomial: $c_2 p^2 + c_1 p + c_0$.  
Since $c_2 = 0$ for all generators, the problem reduces to a **linear program (LP)**.

In [ ]:
gen_data_mat = pd.read_csv("data/gen_data_118-bus.csv")
gencost_mat  = pd.read_csv("data/gencost_data_118-bus.csv")

# Remove the slack bus row so the 53 remaining rows align with gen_df
slack_bus_matpower = int(slack_bus) + 1   # 0-indexed → 1-indexed MATPOWER
slack_idx_mat      = gen_data_mat[gen_data_mat["bus"] == slack_bus_matpower].index

if len(slack_idx_mat) == 0:
    raise ValueError(
        f"Could not find MATPOWER slack bus {slack_bus_matpower} in gen_data_118-bus.csv."
    )

gencost_no_slack = gencost_mat.drop(index=slack_idx_mat[0]).reset_index(drop=True)
c1 = gencost_no_slack["c1"].values  # linear cost coefficient [$/MWh]

assert len(c1) == n_gen, (
    f"Cost vector length ({len(c1)}) != number of generators ({n_gen}). "
    "Check slack removal logic."
)

print(f"Cost vector size: {len(c1)}")
print(f"Generators with c1 > 0: {(c1 > 0).sum()}")
gencost_no_slack[["c2", "c1", "c0"]].head(10)

## 4. Assemble Nodal Vectors

Build the load vector $p_d$ and the generator-to-bus incidence matrix $C_g$.

In [ ]:
# p_load[i] = total active load at non-slack bus i (pu)
p_load = np.zeros(num_ns_buses)
for k, bus in enumerate(load_buses):
    if bus in bus_to_col:
        p_load[bus_to_col[bus]] += load_p_pu[k]

# Cg[i, g] = 1 if generator g is connected to non-slack bus i
Cg = np.zeros((num_ns_buses, n_gen))
for g, bus in enumerate(gen_buses):
    if bus in bus_to_col:
        Cg[bus_to_col[bus], g] = 1.0

print(f"Total load (non-slack buses) : {p_load.sum() * base_mva:.2f} MW")
print(f"Cg shape                     : {Cg.shape}")

## 5. Build the Gurobi Model

In [ ]:
model = gp.Model("DC-OPF_118bus")
model.setParam("OutputFlag", 1)  # set to 0 to silence solver log

In [ ]:
# --- Decision variables ---

# p[g]: active power output of generator g (pu)
p = model.addVars(n_gen, lb=gen_pmin_pu, ub=gen_pmax_pu, name="p")

# theta[i]: voltage angle at non-slack bus i (rad) — unbounded
theta = model.addVars(num_ns_buses, lb=-GRB.INFINITY, ub=GRB.INFINITY, name="theta")

# f[l]: active power flow on branch l (pu)
f = model.addVars(num_branches, lb=-GRB.INFINITY, ub=GRB.INFINITY, name="f")

In [ ]:
# --- Objective: minimize total generation cost [$/h] ---
# c1 [$/MWh] * p[g] [pu] * base_mva [MW/pu] = cost [$/h]
model.setObjective(
    gp.quicksum(c1[g] * p[g] * base_mva for g in range(n_gen)),
    GRB.MINIMIZE
)

In [ ]:
# --- Constraint 1: Nodal power balance at each non-slack bus ---
#   sum_g Cg[i,g]*p[g]  -  p_load[i]  =  sum_j B'[i,j]*theta[j]   for all i
for i in range(num_ns_buses):
    gen_injection = gp.quicksum(Cg[i, g] * p[g] for g in range(n_gen))
    flow_balance  = gp.quicksum(B_prime[i, j] * theta[j] for j in range(num_ns_buses))
    model.addConstr(
        gen_injection - p_load[i] == flow_balance,
        name=f"balance_{non_slack_cols[i] + 1}"
    )

In [ ]:
# --- Constraint 2: DC power flow definition ---
#   f[l] = b_l * sum_i A[l,i] * theta[i]   for all l
for l in range(num_branches):
    flow_expr = gp.quicksum(B_diag[l] * A[l, i] * theta[i] for i in range(num_ns_buses))
    model.addConstr(f[l] == flow_expr, name=f"flow_def_{branch_labels[l]}")

In [ ]:
# --- Constraint 3: Branch thermal limits ---
#   -F_max[l] <= f[l] <= F_max[l]   for all l
for l in range(num_branches):
    model.addConstr(f[l] >= -branches_rate_pu[l], name=f"flow_lb_{branch_labels[l]}")
    model.addConstr(f[l] <=  branches_rate_pu[l], name=f"flow_ub_{branch_labels[l]}")

## 6. Solve

In [ ]:
model.optimize()

## 7. Results

In [ ]:
if model.status == GRB.OPTIMAL:
    total_gen = sum(p[g].X * base_mva for g in range(n_gen))

    print("="*60)
    print("  DC-OPF SOLUTION — IEEE 118-Bus System")
    print("="*60)
    print(f"  Total generation cost : ${model.ObjVal:,.2f} /h")
    print(f"  Total load            : {p_load.sum() * base_mva:.2f} MW")
    print(f"  Total generation      : {total_gen:.2f} MW")

elif model.status == GRB.INFEASIBLE:
    print("Model is INFEASIBLE. Computing IIS...")
    model.computeIIS()
    model.write("dcopf_infeasible.ilp")
    print("IIS written to dcopf_infeasible.ilp")

else:
    print(f"Solver status: {model.status}")

In [ ]:
if model.status == GRB.OPTIMAL:
    # Active generators
    results_gen = []
    for g in range(n_gen):
        pg_mw = p[g].X * base_mva
        if pg_mw > 0.01:
            results_gen.append({
                "gen_idx":     g,
                "bus":         int(gen_buses[g]) + 1,   # MATPOWER 1-indexed
                "p_mw":        round(pg_mw, 2),
                "pmax_mw":     round(gen_pmax_pu[g] * base_mva, 2),
                "cost_$/MWh":  round(c1[g], 4),
            })

    df_gen = pd.DataFrame(results_gen)
    print(f"Active generators: {len(df_gen)}")
    df_gen

In [ ]:
if model.status == GRB.OPTIMAL:
    # Congested branches (loading >= 99%)
    results_branch = []
    for l in range(num_branches):
        fl_mw   = f[l].X * base_mva
        fmax_mw = branches_rate_pu[l] * base_mva
        loading = abs(fl_mw) / fmax_mw * 100 if fmax_mw > 0 else 0
        if loading >= 99.0:
            results_branch.append({
                "branch":      branch_labels[l],
                "from":        int(branches_from[l]) + 1,
                "to":          int(branches_to[l])   + 1,
                "f_mw":        round(fl_mw, 2),
                "fmax_mw":     round(fmax_mw, 2),
                "loading_%":   round(loading, 1),
            })

    df_branch = pd.DataFrame(results_branch)
    if df_branch.empty:
        print("No congested branches.")
    else:
        print(f"Congested branches: {len(df_branch)}")
        df_branch